# تدريب YOLO على كشف الأحرف (OCR)

**الهدف:** تدريب نموذج YOLO ثاني يكتشف **كل حرف على حدة** في اللوحة (27 فئة من 0 إلى 26).

**كيف يعمل الـ OCR:**
1. كاشف اللوحة يحدد موقع اللوحة (موجود لدينا من المرحلة السابقة)
2. نقص اللوحة من الصورة
3. هذا النموذج يكتشف كل حرف داخل اللوحة وموقعه
4. نرتّب الأحرف حسب الموقع (يسار → يمين، فوق → تحت)
5. نحوّل class IDs إلى أحرف فعلية باستخدام character_map.json

**Dataset:**
- 473 صورة تدريب
- 88 صورة validation
- 31 صورة اختبار
- 27 فئة حرفية

## 1. التحقق من GPU + تثبيت المكتبات

In [ ]:
!nvidia-smi | head -10
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 2. رفع dataset الـ OCR

ارفع ملف `ocr_dataset.zip` (سيتم تحضيره)

In [ ]:
import os, shutil, zipfile, yaml
from pathlib import Path
from google.colab import files

if os.path.exists('/content/ocr_dataset'):
    shutil.rmtree('/content/ocr_dataset')
os.makedirs('/content/ocr_dataset', exist_ok=True)

print('ارفع ocr_dataset.zip...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/ocr_dataset')

yaml_files = list(Path('/content/ocr_dataset').rglob('data.yaml'))
DATA_YAML = str(yaml_files[0])
dataset_root = Path(DATA_YAML).parent

# تصحيح المسار
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
cfg['path'] = str(dataset_root.resolve())
cfg['train'] = 'train/images'
cfg['val'] = 'valid/images'
cfg['test'] = 'test/images'
with open(DATA_YAML, 'w') as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f'\ndata.yaml: {DATA_YAML}')
for split in ['train', 'valid', 'test']:
    n = len(list((dataset_root/split/'images').glob('*')))
    print(f'  {split}: {n} صورة')

## 3. عرض عينة للتحقق من البيانات

In [ ]:
import matplotlib.pyplot as plt
import cv2, random

random.seed(7)
imgs = list((dataset_root/'train'/'images').glob('*'))
samples = random.sample(imgs, 4)

# 27 لون مختلف للفئات
colors = [(int(255*(i*37 % 256)/255), int(255*(i*73 % 256)/255), int(255*(i*113 % 256)/255)) for i in range(27)]

fig, axes = plt.subplots(2, 2, figsize=(16, 8))
for ax, p in zip(axes.flatten(), samples):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl = dataset_root/'train'/'labels'/(p.stem + '.txt')
    if lbl.exists():
        with open(lbl) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls = int(float(parts[0]))
                    xc, yc, bw, bh = map(float, parts[1:5])
                    x1 = int((xc-bw/2)*w); y1 = int((yc-bh/2)*h)
                    x2 = int((xc+bw/2)*w); y2 = int((yc+bh/2)*h)
                    cv2.rectangle(img, (x1,y1), (x2,y2), colors[cls], 2)
                    cv2.putText(img, str(cls), (x1, max(15,y1-3)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, colors[cls], 2)
    ax.imshow(img); ax.axis('off'); ax.set_title(p.name[:30])
plt.tight_layout(); plt.show()

## 4. تدريب YOLO على كشف الأحرف

**ملاحظة:** سنستخدم `yolov8n.pt` (nano) لأن المهمة أبسط (صور لوحات صغيرة) والـ dataset أصغر.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=DATA_YAML,
    epochs=80,           # أكثر من detector لأن المهمة أصعب (27 فئة)
    imgsz=640,
    batch=16,
    patience=25,
    project='/content/runs',
    name='ocr_chars_v1',
    exist_ok=True,
    verbose=True,
    # augmentation معتدل (نتجنب augmentations تشوّه الأحرف)
    mosaic=0.5,
    mixup=0.0,           # لا نريد mixup للأحرف
    degrees=5.0,         # دوران خفيف
    translate=0.05,
    scale=0.3,
    fliplr=0.0,          # لا قلب أفقي - يغيّر الحروف!
    flipud=0.0,
    hsv_h=0.01,
    hsv_s=0.5,
    hsv_v=0.3,
    plots=True
)
print('\n✅ التدريب اكتمل')

## 5. التقييم

In [ ]:
best_path = '/content/runs/ocr_chars_v1/weights/best.pt'
best = YOLO(best_path)
m = best.val(data=DATA_YAML)
print(f'mAP@0.5      : {m.box.map50:.4f}')
print(f'mAP@0.5:0.95 : {m.box.map:.4f}')
print(f'Precision    : {m.box.mp:.4f}')
print(f'Recall       : {m.box.mr:.4f}')

In [ ]:
from IPython.display import Image, display
for f in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    p = f'/content/runs/ocr_chars_v1/{f}'
    if os.path.exists(p):
        print(f); display(Image(p))

## 6. تنزيل النموذج

In [ ]:
FINAL = '/content/ocr_chars_detector.pt'
shutil.copy(best_path, FINAL)
print(f'حجم: {os.path.getsize(FINAL)/1e6:.1f} MB')
files.download(FINAL)

# نزّل أيضاً صور تدريب نموذجية لكل فئة (ستحتاجها لبناء character_map)
shutil.make_archive('/content/ocr_training_results', 'zip', '/content/runs/ocr_chars_v1')
files.download('/content/ocr_training_results.zip')
print('\n✅ تم تنزيل النموذج ونتائج التدريب')

## 7. الخطوة التالية: بناء character_map

بعد تنزيل `ocr_chars_detector.pt`:
1. ضعه في مجلد `models/` من المشروع
2. شغّل سكربت `build_character_map.py` على جهازك
3. السكربت سيعرض عينات من كل فئة (class_00 إلى class_26)
4. اكتب أمام كل فئة الحرف الفعلي الذي تراه
5. النتيجة تُحفظ في `character_map.json`
6. الـ OCR pipeline سيستخدم هذا الملف لتحويل class IDs إلى نص فعلي